# SPECTROGRAM NMF

This section processes an audio file's spectrogram into its Non-negative Matrix Factorization (NMF), and then provides visualizations.

Given an audio file, we can view this audiofile's spectrogram simply as a matrix with non-negative entries. Call this matrix $V$.

Then the NMF is just:

$V = W × H$

where $V, W$ and $H$ are all comprised of strictly non-zero entries.

## Setup

Below we import libraries and set up our audio file for processing.

Feel free to skip the second block in this section if yu're not using google colab.

In [ ]:
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from sklearn.decomposition import NMF

In [ ]:
from google.colab import files

print("upload audio file:")
uploaded = files.upload()

# Get the filename of the uploaded file
audio_file = list(uploaded.keys())[0]
print("\n File uploaded successfully")

## Compute Spectogram ($V$)

We use the Short-Time Fourier Transform to compute the spectrogram of the original audio, and call that spectrogram $V$.

### Calculate $V$

In [ ]:
# Load audio
y, sr = librosa.load(audio_file, sr=None)
duration = len(y) / sr

print(f"Audio loaded:")
print(f"  - Duration: {duration:.2f} seconds")
print(f"  - Sample rate: {sr} Hz")

# Compute the STFT
# This gives us the spectrogram matrix V
n_fft = 2048
hop_length = 512

D = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
V = np.abs(D)  # Take magnitude (all non-negative values)

print(f"\nSpectrogram matrix V shape: {V.shape}")
print(f"  - Frequency bins: {V.shape[0]}")
print(f"  - Time frames: {V.shape[1]}")

### Plot a "heatmap" of the spectrogram

In [ ]:
# Plot the spectrogram of the original audio

plt.figure(figsize=(8, 6))
V_db = librosa.amplitude_to_db(V, ref=np.max)
librosa.display.specshow(V_db, sr=sr, hop_length=hop_length,
                         x_axis='time', y_axis='log', cmap='viridis')
plt.colorbar(format='%+2.0f dB')
plt.title('Original Audio Spectrogram (Matrix V)')
plt.tight_layout()
plt.show()


## Perform the NMF

Break $V$ into $W$ and $H$.

### Calculate $W$ and $H$

In [ ]:
# Perform Non-negative Matrix Factorization (NMF)
# Decompose V ≈ W × H
# W = "notes/chords" matrix
# H = "activation" matrix

# Choose number of components
n_components = 88


# Apply NMF
nmf_model = NMF(n_components=n_components, init='nndsvda',
                random_state=42, max_iter=500)
W = nmf_model.fit_transform(V)
H = nmf_model.components_

print(f"  - W (notes) shape: {W.shape} = ({V.shape[0]} frequencies × {n_components} components)")
print(f"  - H (activations) shape: {H.shape} = ({n_components} components × {V.shape[1]} time frames)")

# Sort components
peak_frequencies = np.argmax(W, axis=0)  # Find peak frequency bin for each component
sort_order = np.argsort(peak_frequencies)  # Sort by peak frequency (low to high)

W = W[:, sort_order]  # Reorder columns of W
H = H[sort_order, :]  # Reorder rows of H

# Compute reconstruction
V_reconstructed = W @ H
reconstruction_error = np.linalg.norm(V - V_reconstructed, 'fro') / np.linalg.norm(V, 'fro')
print(f"  - Relative reconstruction error: {reconstruction_error:.4f} ({reconstruction_error*100:.2f}%)")


### Visualize $W$

In [ ]:
# Visualize W

plt.figure(figsize=(8, 6))

# adjust the number below to get a better view
max_freq_bin = min(150, W.shape[0])

plt.imshow(W[:max_freq_bin, :], aspect='auto', origin='lower', cmap='hot', interpolation='nearest')
plt.colorbar(label='Magnitude')
plt.xlabel('Component number')
plt.ylabel('Frequency bin')
plt.title(f'W Matrix Heatmap (First {max_freq_bin} frequency bins)')
plt.tight_layout()
plt.show()

### Visualize $H$

In [ ]:
# Visualize H

plt.figure(figsize=(10, 4))
for i in range(n_components):
    plt.plot(H[i, :], label=f'Component {i+1}', alpha=0.7)

plt.xlabel('Time frame')
plt.ylabel('Activation strength')
plt.title('H Matrix: Time Activations (When Each Component Plays)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Alternative view: H as a heatmap with colors
plt.figure(figsize=(10, 4))
plt.imshow(H, aspect='auto', cmap='hot', interpolation='nearest')
plt.colorbar(label='Activation')
plt.ylabel('Component number')
plt.xlabel('Time frame')
plt.title('H Matrix Heatmap (Activation Pattern)')
plt.tight_layout()
plt.show()

# Binary view: H with threshold (easier to see on/off pattern)
epsilon = 10  # Threshold for "active" vs "inactive"
H_binary = (H > epsilon).astype(int)  # 1 if active, 0 if inactive

plt.figure(figsize=(10, 4))
plt.imshow(H_binary, aspect='auto', cmap='hot', interpolation='nearest')
plt.colorbar(label='Active (1) / Inactive (0)', ticks=[0, 1])
plt.ylabel('Component number')
plt.xlabel('Time frame')
plt.title(f'H Matrix Binary View (Threshold = {epsilon})')
plt.tight_layout()
plt.show()